In [1]:
from instruments.AMI420 import AMI420
from time import sleep
import numpy as np

C:\Users\Ralph Group\anaconda3\lib\site-packages\pyvisa-1.11.3-py3.8.egg\visa.py:13: FutureWarning: The visa module provided by PyVISA is being deprecated. You can replace `import visa` by `import pyvisa as visa` to achieve the same effect.

The reason for the deprecation is the possible conflict with the visa package provided by the https://github.com/visa-sdk/visa-python which can result in hard to debug situations.
  warnings.warn(


In [2]:
def handle_timeout(fail_mode):
    def handle_timeout_decorator(func):
        def func_wrapper(self, *args, **kwargs):
            esp_faliure_counter = 0
            esp_success = False
            FALIURE_LIMIT = 50
            while esp_faliure_counter < FALIURE_LIMIT:
                try:
                    return func(self, *args, **kwargs)
                except pyvisa.errors.VisaIOError:
                    esp_faliure_counter += 1
                    log.warning("failed {mode}, at count {count} of {max}".format(
                        mode=fail_mode,
                        count=esp_faliure_counter,
                        max=FALIURE_LIMIT
                        )
                    )
                    log.info("Clearing GPIB connection %s"%self.name)
                    self.adapter.manager.visalib.clear(self.adapter.connection.session)
                    continue
                esp_success = True # only runs if we beat try statement
                break
            if not esp_success:
                raise RuntimeError("Not able to successfully communicate with magnet")
        return func_wrapper
    return handle_timeout_decorator

# NOTE: magnet is controlled with field

class vectorMagnetBase(AMI420):
    """This is the meta class for messing with a single axis of the vector
    magnet in H8"""

    DELAY = 0.05

    def __init__(self, resourceName, coil_constant, current_limit, axis, field_ramp_rate=0, stability=0, **kwargs):
        """Setting up power supply/controller parameters and setting to remote
        mode"""
        delay = 0.05
        super().__init__(resourceName, **kwargs)
        self.name = "Vector Magnet Vertical Z Axis"

        #Using SI units for Voltage, current, field and ramp rate
        self.field_units = 'tesla'
        self.ramp_rate_units ='seconds'

        print('set units')
        sleep(delay)

        #Setting the min, max output paramters for AMI4Q05100PS
        self.current_minimum = -100
        self.current_maximum = 100
        self.voltage_maximum = 5
        self.voltage_minimum = -5

        print('set current voltage max min')
        sleep(delay)

        #Stability setting should be close to zero when magnet is connected to
        #circuit. If testing the supplies without magnet, use 100%
        #self.stability = stability

        #print('set stability')
        #sleep(delay)

        #Value taken from Manual, also referred to as field to current ratio
        self.coil_constant = coil_constant #Telsa/Amp

        print('set coil constant')
        sleep(delay)

        #Setting the max current to attain a max field of 6.045705 T,
        #~1T less than the maximum rated field
        self.magnet_current_limit = current_limit #Amps

        print('set current limit')
        sleep(delay)

        #Fixing the ramp rate, calculated assuming L=7.8 Henries
        #Max ramp rate = 0.0861 T/sec for max 5V from power supply
        #Setting it to be 0.043 T/sec which corresponds to 320mA/sec which is
        #within ramp rate range for AMI420
        #self.field_ramp_rate = field_ramp_rate #T/sec CHANGED FROM 0.043 T/sec

        #print('set field ramp rate')
        #sleep(delay)

        #What does this do?
        self.auto_quench_detect = True

        print('set auto quench detect')
        sleep(delay)

        #Calculating the field limits given the coil constant and magnet
        #current limit
        field_limit = self.coil_constant*self.magnet_current_limit

        print('read successfully')
        self._fieldlims = [-1*field_limit, field_limit]

        print('set field limits and done')
        self.axis = axis

    """This function sets the field to given setpoint"""
    @handle_timeout("setting field")
    def setField(self, nfield):
        if (nfield < self._fieldlims[0] or nfield > self._fieldlims[1]):
            log.warning(f"""Field setpoint of {nfield} is too high for {self.axis} magnet.
                        Staying at previous setpoint""")
            log.info("%s" %self.state)

        else:
            self.set_ramp_mode()
            self.field_setpoint = nfield

    """Reads the field in the magnet coils"""
    @handle_timeout("getting field")
    def getField(self):
        return self.magnet_field

    field = property(getField, setField)

    """Reads the magnet voltage"""
    @handle_timeout("getting voltage")
    def getMagVoltage(self):
        return self.magnet_voltage

    @handle_timeout("ramping")
    def is_ramping(self):
        try:
            return self.state == 'Ramping'
        except ValueError:
            # with open(r'C:\Users\Ralph Group\Documents\Data\weird_error_log.txt', 'a') as f:
            #     f.write("Failed at STATE?\n")
            #     f.write('STATE read is: '+self.adapter.connection.ask('STATE?')+'\n')
            #     f.write('IDN read is: '+self.adapter.connection.ask('*IDN?')+'\n\n')
            raise ValueError("Bad return from state")

    @handle_timeout("holding")
    def is_holding(self):
        return self.state == 'Holding'

    @handle_timeout("zeroing")
    def is_zeroing(self):
        return self.state == 'Zeroing'

    @handle_timeout("Quench detected")
    def is_quenched(self):
        return self.state == 'Quench'

    @handle_timeout("Paused")
    def is_paused(self):
        return self.state == 'Paused'

    # brings current to zero and ensures in local mode
    @handle_timeout("Shutdown")
    def shutdown(self):
        """ Ensures the magnet is set to zero field """
        super().shutdown()
        log.info("Shutting down the %s magnet"%self.axis)
        self.field = 0. # turn field off
        sleep(0.1)
        self.local() #Can this be done manually from frontpanel?

class vectorMagnetX(vectorMagnetBase):
    """
    The x component of the vector magnet
    """

    def __init__(self, resourceName, **kwargs):
        super().__init__(
            resourceName,
            coil_constant = 0.018891, #T/A TODO: check
            current_limit = 48, # A TODO check, is =0.9T
            stability = 0, # %, does nothing for now
            field_ramp_rate = 0.0043, # T/s, does nothing for now
            axis = 'X',
            **kwargs
        )
        self.name = "Vector Magnet X Axis"

class vectorMagnetY(vectorMagnetBase):
    """
    The y component of the vector magnet
    """

    def __init__(self, resourceName, **kwargs):
        super().__init__(
            resourceName,
            coil_constant = 0.018891, #T/A TODO: check
            current_limit = 48, # A TODO check, is =0.9T
            stability = 0, # %, does nothing for now
            field_ramp_rate = 0.0043, # T/s, does nothing for now
            axis='Y',
            **kwargs
        )
        self.name = "Vector Magnet Y Axis"
        
class vectorMagnetZ(vectorMagnetBase):
    """
    The z component of the vector magnet
    """

    def __init__(self, resourceName, **kwargs):
        super().__init__(
            resourceName,
            coil_constant = 0.134349, #T/A
            current_limit = 45, # A TODO check, is =6T
            stability=50, # %, does nothing for now
            field_ramp_rate=0.0043, # T/s, does nothing for now
            axis = 'Z',
            **kwargs
        )
        self.name = "Vector Magnet Z Axis"
        
class vectorMagnetFull:
    """
    Class to control all three axes of the vector magnet simultaneously.
    Uses the usual physics parameterization of the magnetic field.
    """

    def __init__(self, resourceNameX, resourceNameY, resourceNameZ, **kwargs):
        # QUESTION: should we pass kwargs to all three? I don't think there's
        # a good way to do this...
        self.magnet_x = vectorMagnetX(resourceNameX, **kwargs)
        self.magnet_y = vectorMagnetY(resourceNameY, **kwargs)
        self.magnet_z = vectorMagnetZ(resourceNameZ, **kwargs)

        self._x_field = self.magnet_x.field
        self._y_field = self.magnet_y.field
        self._z_field = self.magnet_z.field

        # limit such that below this field change the magnet does not actually change field,
        # to limit commands sent to the magnet
        self._field_difference_cutoff = 1e-5 # 0.1 G

        # TODO: should we reset the current limit of the z magnet or just
        # trust that the checking in this class will always be OK?

        self._field_mag_lim = 0.92

        self._B_sign = 1

    def set_field_polar(self, B, phi, theta):
        """
        Sets the field, accepting polar coordinates.
        """

        log.info('Setting to %g %g %g'%(B,phi,theta))
        phi = phi*np.pi/180
        theta = theta*np.pi/180

        Bx = B*np.cos(phi)*np.sin(theta)
        By = B*np.sin(phi)*np.sin(theta)
        Bz = B*np.cos(theta)

        if np.sqrt(Bx*Bx + By*By + Bz*Bz) > self._field_mag_lim: #np.sqrt returns positive square root
            log.error("A large field of %g was requested"%np.sqrt(Bx*Bx + By*By + Bz*Bz))
            raise ValueError("Large field requested! Limit is %g"%self._field_mag_lim)

        if B < 0:
            self._B_sign = -1
        else:
            self._B_sign = 1

        # if not np.isclose(Bx, self._x_field, atol=self._field_difference_cutoff, rtol=0):
        self.magnet_x.field = Bx
            # log.info("X change too small")
        # if not np.isclose(By, self._y_field, atol=self._field_difference_cutoff, rtol=0):
        self.magnet_y.field = By
            # log.info("Y change too small")
        # if not np.isclose(Bz, self._z_field, atol=self._field_difference_cutoff, rtol=0):
        self.magnet_z.field = Bz
            # log.info("Z change too small")

    def get_field_polar(self):
        """
        Returns the field in polar coordinates in the standard Physics parameterization
        in the order (B, phi, theta)
        """

        Bx = self.magnet_x.field
        By = self.magnet_y.field
        Bz = self.magnet_z.field

        B = self._B_sign * np.sqrt(Bx**2 + By**2 + Bz**2)
        ang_sign_offset = 0 if self._B_sign > 0 else 180
        phi = (np.arctan2(self._B_sign*By, self._B_sign*Bx)*180/np.pi + ang_sign_offset) % 360
        theta = (np.arctan2(np.sqrt(Bx**2 + By**2), self._B_sign*Bz)*180/np.pi+ ang_sign_offset) % 360

        return B, phi, theta


    def check_field_polar(self, B, phi, theta, RTOL):
        """Checks the current field value to make sure it is within tolerance of setpoint"""
        phi = phi*np.pi/180
        theta = theta*np.pi/180

        Bx_set = B*np.cos(phi)*np.sin(theta)
        By_set = B*np.sin(phi)*np.sin(theta)
        Bz_set = B*np.cos(theta)


        Bx_current = self.magnet_x.field
        By_current = self.magnet_y.field
        Bz_current = self.magnet_z.field

        if not np.isclose(Bx_set,Bx_current, rtol=RTOL) and not np.isclose(By_set,By_current,rtol=RTOL) and not np.isclose(Bz_set, Bz_current, rtol=RTOL):
            log.info("Field is not close to the setpoint")
            return False
        else:
            log.info("field is close to the setpoint")
            return True

    def set_field_cartesian(self, Bx, By, Bz):
        """
        Sets the field using a cartesian basis
        """

        if np.sqrt(Bx*Bx + By*By + Bz*Bz) > self._field_mag_lim: #np.sqrt returns positive square root
            log.error("A large field of %g was requested"%np.sqrt(Bx*Bx + By*By + Bz*Bz))
            raise ValueError("Large field requested! Limit is %g"%self._field_mag_lim)

        # if not np.isclose(Bx, self._x_field, atol=self._field_difference_cutoff, rtol=0):
        self.magnet_x.field = Bx
        # if not np.isclose(By, self._y_field, atol=self._field_difference_cutoff, rtol=0):
        self.magnet_y.field = By
        # if not np.isclose(Bz, self._z_field, atol=self._field_difference_cutoff, rtol=0):
        self.magnet_z.field = Bz


    def get_field_cartesian(self):
        """
        Returns the cartesian parameterization of the field in the order X, Y, Z.
        """

        return self.magnet_x.field, self.magnet_y.field, self.magnet_z.field



    def check_field_cartesian(self, Bx_set, By_set, Bz_set, RTOL):
        """Checks the current field value to make sure it is within tolerance of setpoint """
        Bx_current = self.magnet_x.field
        By_current = self.magnet_y.field
        Bz_current = self.magnet_z.field

        if not np.isclose(Bx_set,Bx_current, rtol=RTOL) and not np.isclose(By_set,By_current,rtol=RTOL) and not np.isclose(Bz_set, Bz_current, rtol=RTOL):
            log.info("Field is not close to the setpoint")
            return False
        else:
            log.info("field is close to the setpoint")
            return True

    def is_ramping(self):
        return self.magnet_x.is_ramping() or self.magnet_y.is_ramping() or self.magnet_z.is_ramping()

    def is_holding(self):
        return self.magnet_x.is_holding() or self.magnet_y.is_holding() or self.magnet_z.is_holding()

    def is_zeroing(self):
        return self.magnet_x.is_zeroing() or self.magnet_y.is_zeroing() or self.magnet_z.is_zeroing()

    def is_quenched(self):
        return self.magnet_x.is_quenched() or self.magnet_y.is_quenched() or self.magnet_z.is_quenched()

    def is_paused(self):
        return self.magnet_x.is_paused() or self.magnet_y.is_paused() or self.magnet_z.is_paused()

    def shutdown(self):
        """
        Shuts down each of the magnets individually
        """
        log.info("Shutting down all of the magnets")
        self.magnet_x.shutdown()
        self.magnet_y.shutdown()
        self.magnet_z.shutdown()

In [7]:
magnet = vectorMagnetZ("GPIB::24")

C:\Users\Ralph Group\anaconda3\lib\site-packages\pyvisa-1.11.3-py3.8.egg\pyvisa\highlevel.py:3352: FutureWarning: get_instrument is deprecated and will be removed in 1.12, use open_resource instead.
  warnings.warn(


set units
set current voltage max min
set coil constant
set current limit
set auto quench detect
read successfully
set field limits and done


In [8]:
def setField(field):
    magnet.field = field
    print("waiting till field is set to setpoint")
    sleep(0.1)
    while magnet.is_ramping():
        sleep(2)
        print("Magnet is ramping")

    while not np.isclose(field, magnet.field, 5e-3):
        sleep(0.1)

    print("Field reached")

In [22]:
30/np.sqrt(3)

17.320508075688775

In [26]:
setField(0)

waiting till field is set to setpoint
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet is ramping
Magnet i

In [9]:
Fullmagnet = vectorMagnetFull("GPIB::26", "GPIB::25", "GPIB::24")

set units
set current voltage max min
set coil constant
set current limit
set auto quench detect
read successfully
set field limits and done
set units
set current voltage max min
set coil constant
set current limit
set auto quench detect
read successfully
set field limits and done
set units
set current voltage max min
set coil constant
set current limit
set auto quench detect
read successfully
set field limits and done


In [3]:
import pyvisa as visa

In [28]:
Fullmagnet.get_field_cartesian()

(-1e-05, 0.0, 5e-05)

In [27]:
Fullmagnet.set_field_cartesian(0.,0.,0.)

In [1]:
from pymeasure.instruments.signalrecovery import DSP7265

C:\Users\Ralph Group\anaconda3\lib\site-packages\pyvisa-1.11.3-py3.8.egg\visa.py:13: FutureWarning: The visa module provided by PyVISA is being deprecated. You can replace `import visa` by `import pyvisa as visa` to achieve the same effect.

The reason for the deprecation is the possible conflict with the visa package provided by the https://github.com/visa-sdk/visa-python which can result in hard to debug situations.
  warnings.warn(


In [1]:
from scanning import ANC150

C:\Users\Ralph Group\anaconda3\lib\site-packages\pyvisa-1.11.3-py3.8.egg\visa.py:13: FutureWarning: The visa module provided by PyVISA is being deprecated. You can replace `import visa` by `import pyvisa as visa` to achieve the same effect.

The reason for the deprecation is the possible conflict with the visa package provided by the https://github.com/visa-sdk/visa-python which can result in hard to debug situations.
  warnings.warn(


In [3]:
stepper = ANC150("COM3")

In [1]:
from pymeasure.instruments.keithley import Keithley2400, Keithley2182A

C:\Users\Ralph Group\anaconda3\lib\site-packages\pyvisa-1.11.3-py3.8.egg\visa.py:13: FutureWarning: The visa module provided by PyVISA is being deprecated. You can replace `import visa` by `import pyvisa as visa` to achieve the same effect.

The reason for the deprecation is the possible conflict with the visa package provided by the https://github.com/visa-sdk/visa-python which can result in hard to debug situations.
  warnings.warn(


In [2]:
sm = Keithley2400("GPIB::17")

C:\Users\Ralph Group\anaconda3\lib\site-packages\pyvisa-1.11.3-py3.8.egg\pyvisa\highlevel.py:3352: FutureWarning: get_instrument is deprecated and will be removed in 1.12, use open_resource instead.
  warnings.warn(


In [3]:
sm.source_current

0.0

In [5]:
for i in range(5):
    print(i)

0
1
2
3
4


In [2]:
import sys
module_dir = r"D:\Github\SagnacOperatingSys\VecMagSagnac_control"
sys.path.append(module_dir)
from sagnac.custom_instruments import vectorMagnetFullUSB_highZ
mag = vectorMagnetFullUSB_highZ()
mag.connect_highZ()

Connected to APS100 on COM4
Connected to APS100 on COM5
Zeroing X magnet
Zeroing Y magnet
Disconnected from APS100


In [3]:
mag.set_field_highZ(2)


In [4]:
mag.get_field_highZ()

2.0

In [2]:
mag.device_2.set_channel(2)

'2'

In [3]:
mag.device_2.zero_field()

In [19]:
mag.device_2.set_field(0)

In [17]:
mag.device_2.write_command("SWEEP ZERO")

In [15]:
mag.device_2.set_channel(2)

'2'

In [20]:
mag.device_2.get_field()

0.0